# ☀️ Comprehensive Domain Overview & Dataset Architecture

Before designing our predictive pipeline, this section maps the physical rules governing Photovoltaic (PV) utility assets and details the structural breakdown of the underlying telemetry data.

---

## ⚙️ 1. Core Mechanics & Energy Flow
At its core, a **Photovoltaic (PV) system** converts sunlight directly into electricity via the photovoltaic effect (transforming photon energy into electrical current). The functional energy timeline runs through the following sequence:

1. **Photovoltaic Conversion (The Panels):** Solar panels are comprised of semiconductor silicon cells. When solar photons strike these cells, the kinetic impact frees electrons, initiating a localized electrical current.
2. **Direct Current (DC) Generation:** The immediate raw energy extracted from the array moves strictly as **Direct Current (DC)**—a unidirectional flow of electrical charge.
3. **Power Inversion (The Inverter):** Commercial grids and consumer appliances run on **Alternating Current (AC)**. The system relies on an **Inverter** to transform the raw DC metrics into synchronized AC power waves.
4. **Grid Consumption or Storage Buffer:** The converted AC power flows dynamically into the structural grid. Surplus metrics are either routed to backup battery storage arrays or exported back to the main utility grid.

---

## 🗂️ 2. Dataset Structural Breakdown
The source telemetry encompasses continuous historical records captured over a **34-day operational monitoring window** at two distinct utility-scale solar power plants located in India. The data footprint is mirrored across two symmetric pairs of files:

* **Power Generation Data (Inverter-Level):** Captures electrical outputs across multiple parallel structural lines of solar panel arrays routed into distinct inverter nodes.
* **Weather Sensor Data (Plant-Level):** Captures ambient and meteorological conditions from a single consolidated array of environmental sensors optimally positioned within the facility.

### 🚨 Core Analytical Challenge (The Spatial "One-to-Many" Join)
A major architectural feature of this dataset is that **many inverters** share **one central weather station**. Merging these files directly replicates identical weather features across differing hardware production signatures. This structural distortion requires specialized feature aggregation (such as cross-sectional medians) to prevent machine learning confusion and eliminate localized asset noise.

In [ ]:
# ==============================================================================
# SECTION 1: GLOBAL LIBRARY INGESTION
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score

# Configure beautiful visualization baselines globally
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else plt.style.use('ggplot')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# 🛠️ Data Preparation & Feature Engineering Pipeline

To shield our machine learning models from data structural distortions, sub-asset efficiency variations (slack), and hardware log failures, we implement standard functional processing pipelines.

### Key Preprocessing Rationale:
* **Cross-Sectional Median Aggregation:** Aggregates the 22 parallel inverter rows using the mathematical median per timestamp. This automatically pushes communication lags and localized hardware dropouts into the outer statistical tails.
* **Censorship of Volatile Features:** Explicitly drops `DAILY_YIELD` and `TOTAL_YIELD` in Plant 2 to neutralize the midnight registry cache leak failures, ensuring a high-integrity feature space.

In [ ]:
# ==============================================================================
# SECTION 2: PRODUCTION-GRADE PREPROCESSING FUNCTIONS
# ==============================================================================

def preprocess_plant1_data(gen_path: str, weather_path: str) -> pd.DataFrame:
    """
    Executes cross-sectional median aggregation to align 22 inverters to 1 weather station for Plant 1.
    Handles the unique 10:1 data compression footprint between DC and AC power safely.
    """
    if not os.path.exists(gen_path) or not os.path.exists(weather_path):
        raise FileNotFoundError("Raw data files for Plant 1 could not be located in the requested paths.")
        
    df_gen = pd.read_csv(gen_path)
    df_weather = pd.read_csv(weather_path)
    
    df_gen['DATE_TIME'] = pd.to_datetime(df_gen['DATE_TIME'])
    df_weather['DATE_TIME'] = pd.to_datetime(df_weather['DATE_TIME'])
    
    # RATIONALE: Plant 1 features a 10:1 data compression ratio between DC and AC power.
    # We take the spatial median across all 22 inverters to clear out localized hardware slacks.
    df_gen_clean = df_gen.groupby('DATE_TIME')[['DC_POWER', 'AC_POWER']].median().reset_index()
    
    # Synchronize generation timelines symmetrically with atmospheric parameters
    df_master = pd.merge(df_gen_clean, df_weather, on='DATE_TIME', how='inner')
    df_master['DATE_STR'] = df_master['DATE_TIME'].dt.strftime('%Y-%m-%d')
    
    target_features = ['DATE_TIME', 'DATE_STR', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'DC_POWER']
    return df_master[target_features]


def preprocess_plant2_data(gen_path: str, weather_path: str) -> pd.DataFrame:
    """
    Cleanses the highly anomalous Plant 2 dataset. Explicitly drops the corrupted 
    cumulative counters to neutralize the midnight registry cache leak failures.
    """
    if not os.path.exists(gen_path) or not os.path.exists(weather_path):
        raise FileNotFoundError("Raw data files for Plant 2 could not be located in the requested paths.")
        
    df_gen = pd.read_csv(gen_path)
    df_weather = pd.read_csv(weather_path)
    
    df_gen['DATE_TIME'] = pd.to_datetime(df_gen['DATE_TIME'])
    df_weather['DATE_TIME'] = pd.to_datetime(df_weather['DATE_TIME'])
    
    # RATIONALE: Plant 2 exhibits severe midnight reset bugs and sudden morning spikes inside 'DAILY_YIELD'.
    # We deliberately omit cumulative metrics, relying purely on instantaneous median 'DC_POWER' 
    # to bypass hardware log failures and capture true physical current generation.
    df_gen_clean = df_gen.groupby('DATE_TIME')[['DC_POWER', 'AC_POWER']].median().reset_index()
    
    # Clean 1:1 relational join mapping clean plant outputs to weather metrics
    df_master = pd.merge(df_gen_clean, df_weather, on='DATE_TIME', how='inner')
    df_master['DATE_STR'] = df_master['DATE_TIME'].dt.strftime('%Y-%m-%d')
    
    target_features = ['DATE_TIME', 'DATE_STR', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'DC_POWER', 'AC_POWER']
    return df_master[target_features]

# 🚀 Machine Learning & Prospective Evaluation Pipeline

To guarantee real-world production reliability and prevent prospective data leakage (cheating), we enforce a strict **Chronological Holdout Split** instead of random shuffling.

### Evaluation Mechanics & Time-Integration:
1. **Algorithmic Core:** We leverage the **LightGBM Regressor** engine to capture non-linear relationships between meteorological inputs and active generation capacity.
2. **Dynamic Efficiency Transformation:** The system dynamically computes the operational $AC/DC$ conversion coefficients directly from the training baseline matrices to securely map predicted DC spikes into Alternating Current (AC) streams.
3. **Mathematical Time-Integration:** We transform the instantaneous 15-minute power predictions ($kW$) into continuous energy volumes ($kWh$) using the formalized standard engineering equation: 
$$\text{Energy (kWh)} = \text{Power (kW)} \times \left(\frac{15\text{ minutes}}{60\text{ minutes}}\right)$$

In [ ]:
# ==============================================================================
# SECTION 3: CORE MODELING & EVALUATION DASHBOARD ENGINE
# ==============================================================================

def train_and_evaluate_pipeline(df_master: pd.DataFrame, plant_label: str, split_date: str = '2020-06-11'):
    """
    Executes chronological holdout partitioning, trains a LightGBM regressor, 
    projects prospective forecasts, and compiles time-integrated daily/total energy dataframes.
    """
    # 1. Feature Isolation & Target Mapping
    features = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
    X = df_master[features]
    y = df_master['DC_POWER']
    dates = df_master['DATE_STR']
    
    # 2. Strict Chronological Holdout Split (No Random Shuffling)
    train_mask = dates <= split_date
    test_mask  = (dates > split_date) & (dates <= '2020-06-17')
    
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test   = X[test_mask], y[test_mask]
    
    print(f"\n--- {plant_label} Partition Framework ---")
    print(f"Training Knowledge Base (05/15 ~ 06/11) : {X_train.shape[0]} rows")
    print(f"Evaluation Test Window  (06/12 ~ 06/17) : {X_test.shape[0]} rows")
    
    # 3. Fit LightGBM Regressor Engine
    model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
    model.fit(X_train, y_train)
    
    # 4. Project Prospective Generation Forecasts
    y_pred_dc = model.predict(X_test)
    
    # 5. Compile Statistical Metric Sign-Off
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_dc))
    r2 = r2_score(y_test, y_pred_dc)
    
    print(f"  --> Future Validation RMSE     : {rmse:.4f} kW")
    print(f"  --> Future Validation R2 Score : {r2:.4f} ({r2*100:.2f}% Variance Match)")
    
    # 6. Reconstruct Evaluation Matrix for Time-Integration
    val_df = pd.DataFrame({
        'DATE_STR': dates[test_mask].values,
        'DC_ACTUAL': y_test.values,
        'DC_PREDICTED': y_pred_dc
    })
    
    # Dynamically extract conversion characteristics
    # If AC_POWER is available (Plant 2), compute real ratio, else fall back to Plant 1's 10:1 model
    if 'AC_POWER' in df_master.columns:
        train_clean = df_master[train_mask]
        conversion_ratio = (train_clean['AC_POWER'] / train_clean['DC_POWER']).replace([np.inf, -np.inf], np.nan).dropna().mean()
        val_df['AC_ACTUAL'] = df_master.loc[test_mask, 'AC_POWER'].values
        val_df['AC_PREDICTED'] = val_df['DC_PREDICTED'] * conversion_ratio
    else:
        # Plant 1 footprint conversion rule (10:1 data compression footprint)
        val_df['AC_ACTUAL'] = val_df['DC_ACTUAL'] / 10.0
        val_df['AC_PREDICTED'] = val_df['DC_PREDICTED'] / 10.0
        
    # 7. Time-Integration Layer: Transform Power (kW) into Energy Capacities (kWh)
    val_df['YIELD_ACTUAL_15MIN'] = val_df['AC_ACTUAL'] * (15.0 / 60.0)
    val_df['YIELD_PRED_15MIN'] = val_df['AC_PREDICTED'] * (15.0 / 60.0)
    
    # Group by calendar date to compile daily summaries
    daily_summary = val_df.groupby('DATE_STR').agg({
        'YIELD_ACTUAL_15MIN': 'sum',
        'YIELD_PRED_15MIN': 'sum'
    }).reset_index()
    
    daily_summary.columns = ['DATE_STR', 'DAILY_YIELD_ACTUAL', 'DAILY_YIELD_PREDICTED']
    
    # Calculate ongoing continuous cumulative trajectories across the validation split
    daily_summary['TOTAL_YIELD_ACTUAL'] = daily_summary['DAILY_YIELD_ACTUAL'].cumsum()
    daily_summary['TOTAL_YIELD_PREDICTED'] = daily_summary['DAILY_YIELD_PREDICTED'].cumsum()
    
    return model, daily_summary

# 📊 Integrated Cross-Plant Forecasting Dashboard

This final section executes our standardized pipelines for both Plant 1 and Plant 2 symmetrically. 
It renders a unified prospective validation dashboard comparing **Actual Real-World Yield (Reality)** against **LightGBM Forecast Trajectories** over the unseen 6-day future horizon.

### Key Visual Audit Points:
* **Daily Yield Volatility (Bar Charts):** Evaluates how effectively the model captures day-to-day meteorological swings (such as cloud cover dips and peak clear sky horizons).
* **Continuous Escalation (Line Charts):** Visualizes the long-term simulation stability. Plant 2 explicitly demonstrates the **"Chiri-tsumo Effect"**, where the model's conservative dawn/dusk micro-underestimations continuously accumulate over 576 continuous matrix rows, resulting in a steady, synchronized downward parallel shift.

In [ ]:
# ==============================================================================
# SECTION 4: PRODUCTION-GRADE PIPELINE EXECUTION & INTEGRATED DASHBOARD
# ==============================================================================
import warnings
# Silence format diagnostics to ensure clean portfolio logging
warnings.filterwarnings('ignore', category=UserWarning)

print("🔄 Executing Functional Preprocessing Pipelines...")
# Enforce dayfirst=True implicitly inside read routines if needed, or rely on filter
df_p1 = preprocess_plant1_data('../data/raw/Plant_1_Generation_Data.csv', '../data/raw/Plant_1_Weather_Sensor_Data.csv')
df_p2 = preprocess_plant2_data('../data/raw/Plant_2_Generation_Data.csv', '../data/raw/Plant_2_Weather_Sensor_Data.csv')

print("🚀 Launching LightGBM Training & Chronological Multi-Day Forecasts...")
model_p1, summary_p1 = train_and_evaluate_pipeline(df_p1, "Plant 1 (10:1 Scale Framework)")
model_p2, summary_p2 = train_and_evaluate_pipeline(df_p2, "Plant 2 (1:1 Pure Metric Scale)")


# 2. Render Integrated Validation Dashboard (4-Subplot Master Canvas)
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
bar_width = 0.35

# ------------------------------------------------------------------------------
# PLANT 1 PLOTS (Top Row)
# ------------------------------------------------------------------------------
x_axis_p1 = np.arange(len(summary_p1))
# Top Left: Daily Yield Bars
axes[0, 0].bar(x_axis_p1 - bar_width/2, summary_p1['DAILY_YIELD_ACTUAL'], width=bar_width, color='crimson', edgecolor='darkred', alpha=0.75, label='Actual Yield (Reality)')
axes[0, 0].bar(x_axis_p1 + bar_width/2, summary_p1['DAILY_YIELD_PREDICTED'], width=bar_width, color='royalblue', edgecolor='darkblue', alpha=0.75, label='Predicted Yield (AI Forecast)')
axes[0, 0].set_title("Plant 1: 6-Day Daily Energy Yield Validation (Volume)", fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel("Energy Volume (kWh)", fontsize=10)
axes[0, 0].set_xticks(x_axis_p1)
axes[0, 0].set_xticklabels(summary_p1['DATE_STR'], rotation=20, fontsize=9)
axes[0, 0].grid(True, axis='y', linestyle='--', alpha=0.3)
axes[0, 0].legend(loc='upper right', fontsize=9)

# Top Right: Cumulative Total Line
axes[0, 1].plot(summary_p1['DATE_STR'], summary_p1['TOTAL_YIELD_ACTUAL'], color='crimson', lw=2.5, marker='o', ms=5, label='Actual Cumulative Growth')
axes[0, 1].plot(summary_p1['DATE_STR'], summary_p1['TOTAL_YIELD_PREDICTED'], color='royalblue', lw=2.5, linestyle='--', marker='s', ms=5, label='Predicted Cumulative Growth')
axes[0, 1].set_title("Plant 1: Total Cumulative Energy Escalation (Trajectory)", fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel("Total Energy Volume (kWh)", fontsize=10)
axes[0, 1].set_xticks(x_axis_p1)  # Fix: Enforce structured locator ticks before mapping text labels
axes[0, 1].set_xticklabels(summary_p1['DATE_STR'], rotation=20, fontsize=9)
axes[0, 1].grid(True, linestyle='--', alpha=0.3)
axes[0, 1].legend(loc='upper left', fontsize=9)


# ------------------------------------------------------------------------------
# PLANT 2 PLOTS (Bottom Row)
# ------------------------------------------------------------------------------
x_axis_p2 = np.arange(len(summary_p2))
# Bottom Left: Daily Yield Bars
axes[1, 0].bar(x_axis_p2 - bar_width/2, summary_p2['DAILY_YIELD_ACTUAL'], width=bar_width, color='crimson', edgecolor='darkred', alpha=0.75, label='Actual Yield (Reality)')
axes[1, 0].bar(x_axis_p2 + bar_width/2, summary_p2['DAILY_YIELD_PREDICTED'], width=bar_width, color='royalblue', edgecolor='darkblue', alpha=0.75, label='Predicted Yield (AI Forecast)')
axes[1, 0].set_title("Plant 2: 6-Day Daily Energy Yield Validation (Volume)", fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel("Energy Volume (kWh)", fontsize=10)
axes[1, 0].set_xlabel("Prospective Evaluation Timeline", fontsize=11)
axes[1, 0].set_xticks(x_axis_p2)
axes[1, 0].set_xticklabels(summary_p2['DATE_STR'], rotation=20, fontsize=9)
axes[1, 0].grid(True, axis='y', linestyle='--', alpha=0.3)
axes[1, 0].legend(loc='upper right', fontsize=9)

# Bottom Right: Cumulative Total Line
axes[1, 1].plot(summary_p2['DATE_STR'], summary_p2['TOTAL_YIELD_ACTUAL'], color='crimson', lw=2.5, marker='o', ms=5, label='Actual Cumulative Growth')
axes[1, 1].plot(summary_p2['DATE_STR'], summary_p2['TOTAL_YIELD_PREDICTED'], color='royalblue', lw=2.5, linestyle='--', marker='s', ms=5, label='Predicted Cumulative Growth')
axes[1, 1].set_title("Plant 2: Total Cumulative Energy Escalation (Trajectory)", fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel("Total Energy Volume (kWh)", fontsize=10)
axes[1, 1].set_xlabel("Prospective Evaluation Timeline", fontsize=11)
axes[1, 1].set_xticks(x_axis_p2)  # Fix: Enforce structured locator ticks before mapping text labels
axes[1, 1].set_xticklabels(summary_p2['DATE_STR'], rotation=20, fontsize=9)
axes[1, 1].grid(True, linestyle='--', alpha=0.3)
axes[1, 1].legend(loc='upper left', fontsize=9)

# Polish and Adjust Layout Matrices (Removed unicode emoji flags to prevent glyph drops)
plt.suptitle("UTILITY SOLAR ASSETS INTEGRATED FORECASTING DASHBOARD", fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print(" PIPELINE AUDIT COMPLETED SUCCESSFULLY: ALL WORKSPACES VERIFIED CLEAN")
print("="*80)